In [2]:
import io
import numpy as np
import pandas as pd

raw_csv = """Date,Record_Count,Total_Orders,Total_Sales,Valid_Records_Rate,Rejected_Records,Missing_Rate,Duplicate_Rate
2025-09-01,1203,661,313993.69,0.983,20,0.0088,0.0012
2025-09-02,1152,633,294243.16,0.9689,35,0.0336,0.005
2025-09-03,1340,737,277589.2,0.9849,20,0.0154,0.0011
2025-09-04,1241,682,300612.84,0.9804,24,0.0301,0.0044
2025-09-05,1121,616,242140.41,0.9816,20,0.0211,0.0028
2025-09-06,1126,619,268310.06,0.9891,12,0.0314,0.001
2025-09-07,1280,704,274171.7,0.9675,41,0.0322,0.0034
2025-09-08,1172,644,293616.7,0.9711,33,0.0323,0.0017
2025-09-09,1231,677,286188.64,0.9754,30,0.0207,0.0072
2025-09-10,1124,618,236649.63,0.9655,38,0.0123,0.008
2025-09-11,1324,728,300548.11,0.9666,44,0.0089,0.0068
2025-09-12,1080,594,295348.24,0.9745,27,0.0104,0.0014
2025-09-13,1177,647,299538.07,0.979,24,0.0114,0.0059
2025-09-14,1294,711,258093.46,0.983,21,0.0322,0.0002
2025-09-15,1153,634,307155.04,0.9678,37,0.0085,0.0023
2025-09-16,1313,722,324779.39,0.9906,12,0.0139,0.0025
2025-09-17,1191,655,254833.38,0.9932,8,0.0172,0.0035
2025-09-18,1317,724,240008.27,0.9662,44,0.0098,0.0032
2025-09-19,1336,734,330588.9,0.9873,16,0.0227,0.0053
2025-09-20,1342,738,290530.27,0.9928,9,0.012,0.005
2025-09-21,1181,649,261638.22,0.9886,13,0.0322,0.0061
2025-09-22,1180,649,247877.09,0.9699,35,0.035,0.0012
2025-09-23,1070,588,313361.07,0.9668,35,0.0315,0.0051
2025-09-24,1158,636,243808.88,0.9674,37,0.0245,0.0019
2025-09-25,1141,627,294904.72,0.9817,20,0.0252,0.0026
2025-09-26,1217,669,307370.46,0.9689,37,0.0165,0.0032
2025-09-27,1229,675,226544.36,0.9676,39,0.0232,0.0077
2025-09-28,1252,688,293558.61,0.967,41,0.0298,0.0055
2025-09-29,1189,653,262542.87,0.9665,39,0.0297,0.0057
2025-09-30,1150,632,277685.77,0.973,31,0.0293,0.002"""

df = pd.read_csv(io.StringIO(raw_csv))

df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = (
    df.dropna(subset=["Date"])
    .drop_duplicates(subset=["Date"], keep="first")
    .sort_values("Date")
    .reset_index(drop=True)
)

num_cols = [
    "Record_Count",
    "Total_Orders",
    "Total_Sales",
    "Valid_Records_Rate",
    "Rejected_Records",
    "Missing_Rate",
    "Duplicate_Rate",
]
for col in num_cols:
  df[col] = pd.to_numeric(df[col], errors="coerce")
  median_val = df[col].median()
  df[col] = df[col].fillna(0 if pd.isna(median_val) else median_val)

non_negative_cols = [
    "Record_Count",
    "Total_Orders",
    "Total_Sales",
    "Rejected_Records",
    "Missing_Rate",
    "Duplicate_Rate",
]
for col in non_negative_cols:
  df[col] = df[col].clip(lower=0)

rate_cols = ["Valid_Records_Rate", "Missing_Rate", "Duplicate_Rate"]
for col in rate_cols:
  df[col] = df[col].clip(lower=0.0, upper=1.0)

int_cols = ["Record_Count", "Total_Orders", "Rejected_Records"]
for col in int_cols:
  df[col] = df[col].round().astype("int64")

df["Avg_Order_Value"] = np.where(
    df["Total_Orders"] > 0, (df["Total_Sales"] / df["Total_Orders"]).round(2), 0.0
)

print("Final Shape:", df.shape)
print("Remaining Missing Values:", int(df.isnull().sum().sum()))
display(df.head(10))

df.to_csv("previous_days_summary_cleaned.csv", index=False)

Final Shape: (30, 9)
Remaining Missing Values: 0


,Date,Record_Count,Total_Orders,Total_Sales,Valid_Records_Rate,Rejected_Records,Missing_Rate,Duplicate_Rate,Avg_Order_Value
0,2025-09-01,1203,661,313993.69,0.9830,20,0.0088,0.0012,475.03
1,2025-09-02,1152,633,294243.16,0.9689,35,0.0336,0.0050,464.84
2,2025-09-03,1340,737,277589.20,0.9849,20,0.0154,0.0011,376.65
3,2025-09-04,1241,682,300612.84,0.9804,24,0.0301,0.0044,440.78
4,2025-09-05,1121,616,242140.41,0.9816,20,0.0211,0.0028,393.09
5,2025-09-06,1126,619,268310.06,0.9891,12,0.0314,0.0010,433.46
6,2025-09-07,1280,704,274171.70,0.9675,41,0.0322,0.0034,389.45
7,2025-09-08,1172,644,293616.70,0.9711,33,0.0323,0.0017,455.93
8,2025-09-09,1231,677,286188.64,0.9754,30,0.0207,0.0072,422.73
9,2025-09-10,1124,618,236649.63,0.9655,38,0.0123,0.0080,382.93
